In [4]:
pip install "gymnasium[classic-control]" stable-baselines3 sb3-contrib

Note: you may need to restart the kernel to use updated packages.


In [10]:
from typing import Optional
import numpy as np
import gymnasium as gym

class ScheduleEnv(gym.Env):
    def __init__(self, num_employees: int = 10, num_days: int = 14):
        super(ScheduleEnv, self).__init__()
        self.action_space = gym.spaces.Discrete(3)
        self.num_employees = num_employees
        self.num_days = num_days
        self.max_consecutive_days = 5
        self.minimum_coverage = 3
        self.max_days_per_year = 23

        obs_size = 2 + (3 * num_employees) + 2
        self.observation_space = gym.spaces.Box(
            low=0,
            high=max(num_days, self.max_days_per_year),
            shape=(obs_size,),
            dtype=np.float32
        )

        self.reset()

    def _get_obs(self):
        obs = np.zeros(self.observation_space.shape, dtype=np.float32)
        
        safe_day = min(self.current_day, self.num_days - 1)
        
        idx = 0
        obs[idx] = self.current_employee;        idx += 1
        obs[idx] = self.current_day;              idx += 1

        obs[idx:idx+self.num_employees] = self.days_worked;          idx += self.num_employees
        obs[idx:idx+self.num_employees] = self.consecutive_days;     idx += self.num_employees
        obs[idx:idx+self.num_employees] = self.last_shift;           idx += self.num_employees

        obs[idx] = self.daily_coverage_T[safe_day];   idx += 1
        obs[idx] = self.daily_coverage_M[safe_day];   idx += 1

        return obs
    
    def _get_info(self):
        info = {
            "current_employee": self.current_employee,
            "current_day": self.current_day,
            "days_worked": self.days_worked.copy(),
            "consecutive_days": self.consecutive_days.copy(),
            "last_shift": self.last_shift.copy(),
            "daily_coverage_T": self.daily_coverage_T.copy(),
            "daily_coverage_M": self.daily_coverage_M.copy()
        }
        return info
    
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)

        self.current_employee = 0
        self.current_day = 0

        self.schedule = np.zeros((self.num_employees, self.num_days), dtype=np.float32)

        self.days_worked = np.zeros(self.num_employees, dtype=np.float32)
        self.consecutive_days = np.zeros(self.num_employees, dtype=np.float32)
        self.last_shift = np.zeros(self.num_employees, dtype=np.float32)
        
        self.daily_coverage_T = np.zeros(self.num_days, dtype=np.float32)
        self.daily_coverage_M = np.zeros(self.num_days, dtype=np.float32)

        observation = self._get_obs()
        info = self._get_info()

        return observation, info
    
    def _calculate_reward(self, emp: int, day: int, action: int) -> int:
        reward = 0

        if action > 0:
            if action == 1 and self.daily_coverage_T[day] < self.minimum_coverage:
                reward += 4
            elif action == 2 and self.daily_coverage_M[day] < self.minimum_coverage:
                reward += 4

            if action == 1 and self.daily_coverage_T[day] >= self.minimum_coverage:
                reward -= 1
            elif action == 2 and self.daily_coverage_M[day] >= self.minimum_coverage:
                reward -= 1
        else:
            if self.consecutive_days[emp] >= self.max_consecutive_days:
                reward += 5

        return reward

    def _check_daily_coverage(self, day: int) -> int:
        reward = 0

        if self.daily_coverage_T[day] == self.minimum_coverage:
            reward += 10
        else:
            reward -= abs(self.minimum_coverage - self.daily_coverage_T[day]) * 5
        
        if self.daily_coverage_M[day] == self.minimum_coverage:
            reward += 10
        else:
            reward -= abs(self.minimum_coverage - self.daily_coverage_M[day]) * 5
        
        return reward

    def _calculate_final_reward(self) -> float:
        reward = 0.0
        
        all_valid = True
        for day in range(self.num_days):
            if (self.daily_coverage_T[day] != self.minimum_coverage or
                self.daily_coverage_M[day] != self.minimum_coverage):
                all_valid = False
                break
        
        if all_valid:
            reward += 100
        
        workload_std = np.std(self.days_worked)
        reward -= workload_std * 3
        
        return reward
    
    def step(self, action: int):
        day = self.current_day
        emp = self.current_employee

        reward = self._calculate_reward(emp, day, action)

        self.schedule[emp, day] = action

        if action > 0:
            self.days_worked[emp] += 1
            self.consecutive_days[emp] += 1
            self.last_shift[emp] = action

            if action == 1:
                self.daily_coverage_T[day] += 1
            else:
                self.daily_coverage_M[day] += 1
        else:
            self.consecutive_days[emp] = 0
            self.last_shift[emp] = 0

        self.current_employee += 1
        if self.current_employee >= self.num_employees:
            reward += self._check_daily_coverage(day)
            
            self.current_employee = 0
            self.current_day += 1

        terminated = self.current_day >= self.num_days

        if terminated:
            reward += self._calculate_final_reward()

        truncated = False

        return self._get_obs(), reward, terminated, truncated, self._get_info()
    
    def render(self):
        shift_map = {0: '-', 1: 'T', 2: 'M'}

        header = "Day\\Emp | " + " | ".join([f"E{e:2d}" for e in range(self.num_employees)]) + " | T  M"
        print(header)
        print("-" * len(header))
        for day in range(self.num_days):
            row = f"{day:6} | " + " | ".join(
                [shift_map[int(self.schedule[emp, day])] for emp in range(self.num_employees)]
            )
            row += f" | {int(self.daily_coverage_T[day]):2d} {int(self.daily_coverage_M[day]):2d}"
            print(row)

    def get_action_mask(self):
        day = self.current_day
        emp = self.current_employee
        
        if day >= self.num_days:
            return np.array([True, False, False])
        
        mask = np.ones(3, dtype=bool)
        
        if self.days_worked[emp] >= self.max_days_per_year:
            mask[1] = False
            mask[2] = False
        
        if self.consecutive_days[emp] >= self.max_consecutive_days:
            mask[1] = False
            mask[2] = False
        
        if day > 0 and self.schedule[emp, day - 1] == 1:
            mask[2] = False
        
        return mask
        
            

In [ ]:
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.common.maskable.utils import get_action_masks

def mask_fn(env):
    return env.get_action_mask()

env = ScheduleEnv(num_employees=10, num_days=31)
env = ActionMasker(env, mask_fn)

# Create the model
# model = MaskablePPO(
#     "MlpPolicy",
#     env,
#     verbose=1,
#     learning_rate=3e-4,
#     n_steps=1024,
#     batch_size=64,
#     n_epochs=10,
# )

# Load existing model
model = MaskablePPO.load("schedule_model", env=env, verbose=1)

model.learn(total_timesteps=200000)
print("Training complete!")

model.save("schedule_model")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Training...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 310      |
|    ep_rew_mean     | 1.19e+03 |
| time/              |          |
|    fps             | 567      |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 1024     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 310         |
|    ep_rew_mean          | 1.17e+03    |
| time/                   |             |
|    fps                  | 365         |
|    iterations           | 2           |
|    time_elapsed         | 5           |
|    total_timesteps      | 2048        |
| train/                  |             |
|    approx_kl            | 0.018099207 |
|    clip_fraction        | 0.0792      |
|    clip_range           | 0.2         |
|    entropy_loss        

In [ ]:
model = MaskablePPO.load("schedule_model")

env = ScheduleEnv(num_employees=10, num_days=31)

obs, info = env.reset()
done = False
total_reward = 0

while not done:
    mask = env.get_action_mask()
    action, _ = model.predict(obs, action_masks=mask, deterministic=True)
    
    obs, reward, terminated, truncated, info = env.step(int(action))
    total_reward += reward
    done = terminated or truncated

print(f"Total reward: {total_reward:.2f}")
print(f"Days worked per employee: {env.days_worked}")
print(f"Workload std: {np.std(env.days_worked):.2f}")
print()

violations = 0
for day in range(env.num_days):
    if env.daily_coverage_T[day] < env.minimum_coverage:
        print(f"Day {day}: T coverage = {int(env.daily_coverage_T[day])} (expected {env.minimum_coverage})")
        violations += 1
    if env.daily_coverage_M[day] < env.minimum_coverage:
        print(f"Day {day}: M coverage = {int(env.daily_coverage_M[day])} (expected {env.minimum_coverage})")
        violations += 1

if violations == 0:
    print("✅ All coverage constraints met!")
else:
    print(f"❌ {violations} coverage violations found")

print()
env.render()

Total reward: 1059.92
Days worked per employee: [23. 23. 23. 23. 22. 20. 12.  9.  8.  7.]
Workload std: 6.69

Day 27: T coverage = 0 (expected 3)
Day 27: M coverage = 0 (expected 3)
Day 28: T coverage = 0 (expected 3)
Day 28: M coverage = 0 (expected 3)
Day 29: T coverage = 0 (expected 3)
Day 29: M coverage = 0 (expected 3)
Day 30: T coverage = 0 (expected 3)
Day 30: M coverage = 0 (expected 3)
❌ 8 coverage violations found

Day\Emp | E 0 | E 1 | E 2 | E 3 | E 4 | E 5 | E 6 | E 7 | E 8 | E 9 | T  M
--------------------------------------------------------------------------
     0 | M | T | M | T | - | - | - | T | M | - |  3  3
     1 | M | T | M | T | M | - | - | - | - | T |  3  3
     2 | M | T | M | T | M | T | - | - | - | - |  3  3
     3 | M | T | M | T | M | - | T | - | - | - |  3  3
     4 | M | T | M | T | M | - | - | T | - | - |  3  3
     5 | - | - | - | - | M | M | T | T | M | T |  3  3
     6 | M | M | M | T | - | T | - | - | - | T |  3  3
     7 | M | M | M | T | T | T | - |